# Challenge 1: Narrative & Visual Consistency in Long-Form Generative Video
### Feasibility test of the Scene-Graph Orchestrator with a real video diffusion model

CSc 8830 Computer Vision. Challenge from Dr. Mohit Bansal's talk.

**Problem.** Video generators make good short clips, but over a multi-scene story they show **character drift**
(faces and clothes change), **object & spatial shift** (things vanish, rooms rearrange), **limited memory** of
earlier scenes, and they need **constant re-prompting**.

**Proposed solution.** A *Scene-Graph Orchestrator* that separates narrative planning from video generation:

`User prompt → LLM narrative planner → persistent world/state graph → control adapter → video generator → consistent video`

**What this notebook does.** It runs the full pipeline on a free Colab GPU, with no training:

| Stage | Here |
|---|---|
| 2. Narrative planner | Gemini (if a `GEMINI_API_KEY` Colab secret is set), else the rule-based planner from the web app |
| 3. State graph | `WorldState` from the web app, which locks appearance and tracks objects, places and relations |
| 4a. Control adapter | state → enriched prompt + negative prompt + fixed seed + **IP-Adapter** reference image (the last frame seen in that place, or the identity anchor from scene 1) |
| 4b. Generator | **AnimateDiff** (Stable Diffusion 1.5 + motion module), a pretrained text-to-video diffusion model |
| Evaluation | **OWLv2** open-vocabulary detector, **DINOv2** identity embeddings, **CLIP**, optical-flow warping error, optional **Gemini** Video-LLM judge |

The same story is generated twice with the same model:
* **Baseline**: each scene prompted with its own sentence and a fresh seed (plain re-prompting).
* **Orchestrator**: each scene conditioned on the persistent world state.

**How to run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (~15-25 min, mostly model
downloads and generation). Optional: add a Colab secret named `GEMINI_API_KEY` (key icon in the left bar) to use
Gemini as the planner and the Video-LLM judge.

Code: `homeworks/challenge_video/` in this repo. The web app's *Challenge 1* pages run the same stages offline.

## 0. Setup

In [ ]:
import os, sys, subprocess

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU found - switch the runtime to T4 GPU.")
!pip -q install "diffusers>=0.30" "transformers>=4.44" accelerate imageio-ffmpeg

REPO = "https://github.com/soonebabu/CV_HomeworkModule.git"
if not os.path.exists("CV_HomeworkModule/homeworks"):
    !git clone -q --depth 1 {REPO}
os.chdir("CV_HomeworkModule")
sys.path.insert(0, os.getcwd())
assert os.path.isdir("homeworks/challenge_video"), "The challenge module is not on GitHub yet - push it first."

# optional: Gemini as planner + Video-LLM judge (Colab secret GEMINI_API_KEY)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass
print("LLM:", "Gemini key found" if os.environ.get("GEMINI_API_KEY") else "none - rule-based planner, no judge")

In [ ]:
import base64, json, zlib
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, Image, display
from PIL import Image as PILImage

from homeworks.challenge_video import (control_adapter, judge, llm, metrics, pipeline, planner, real_eval,
                                       state_graph)
from homeworks.challenge_video.planner import HUMAN_KINDS
from homeworks.hw6_motion.video_io import VideoWriter

pd.set_option("display.max_colwidth", 200)
OUT = "/content/challenge_out" if os.path.isdir("/content") else os.path.abspath("challenge_out")
os.makedirs(OUT, exist_ok=True)

# ---- settings ------------------------------------------------------------------------------------------
STORY = ("Mia, a girl with a red jacket and black hair, walks her brown dog Max in the park. "
         "She sits on a bench next to a yellow ball. Max picks up the ball. "
         "It starts to rain and Mia opens a blue umbrella. They go home to the kitchen. "
         "Mia puts the ball on the table and Max sleeps under the table.")
BASE_MODEL = "emilianJR/epiCRealism"                     # SD 1.5 checkpoint used in the diffusers AnimateDiff docs
MOTION_ADAPTER = "guoyww/animatediff-motion-adapter-v1-5-2"
FRAMES, STEPS, SIZE, GUIDANCE = 16, 25, 512, 7.5
IP_SCALE = 0.6                                          # strength of the visual-memory adapter
STYLE = "high quality photo, cinematic lighting, detailed"
BASE_NEGATIVE = "blurry, low quality, deformed, distorted face, extra limbs, text, watermark"
SEED = zlib.crc32(STORY.encode())

## 1. Stage 2: narrative planner
The story becomes a structured plan: entities (characters and objects, with appearance) and scenes (place,
weather, time, actions, spatial relations).

In [ ]:
plan, plan_info = planner.plan(STORY)
print("Planner:", plan_info["planner"], "" if not plan_info["fallback_reason"] else f"({plan_info['fallback_reason']})")
display(pd.DataFrame([{"id": e["id"], "name": e["name"], "type": e["type"], "kind": e["kind"],
                       "attributes from story": ", ".join(f"{k}={v}" for k, v in e["attributes"].items())}
                      for e in plan["entities"]]))
display(pd.DataFrame([{"#": s["index"], "text": s["text"], "place": s["location"] or "(same)",
                       "actions": "; ".join(f"{a['actor']} {a['verb']} {a['target'] or ''}".strip() for a in s["actions"]),
                       "relations": "; ".join(f"{r['subject']} {r['relation']} {r['object']}" for r in s["relations"])}
                      for s in plan["scenes"]]))

## 2. Stage 3: persistent world / state graph
Appearance the story leaves open is chosen once and then **locked**. Objects stay where they are until moved,
carried objects follow whoever holds them, and place, weather and time carry over between scenes.

In [ ]:
world = state_graph.WorldState(plan)
snaps = [world.apply(sc) for sc in plan["scenes"]]
state_graph.draw_graphs(snaps, f"{OUT}/state_graph.png")
display(Image(f"{OUT}/state_graph.png"))
for s in snaps:
    print(f"scene {s['index']}: {s['env']['location']:8s} visible={s['visible']}  changes={s['changes']}")

## 3. Stage 4a: control adapter
Each scene gets its conditioning from the state graph. Stable Diffusion's text encoder reads at most 77 tokens,
so the adapter packs the most important state first: setting, scene text, each character's locked look, then
objects. The visual memory goes through **IP-Adapter**: the reference image is the last frame generated *in
this place* (spatial memory), or the scene-1 keyframe when entering a new place (identity anchor).

The baseline gets only the scene sentence (plus the same style words) and a new seed per scene.

In [ ]:
from transformers import CLIPTokenizer
tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")


def fit(parts, limit=75):
    # keep adding parts while the prompt still fits in CLIP's context
    out = []
    for p in parts:
        if p and len(tokenizer(", ".join(out + [p])).input_ids) <= limit + 2:
            out.append(p)
    return ", ".join(out)


def look(ent, snap):
    a = ent["attributes"]
    if ent["type"] == "character" and ent["kind"] in HUMAN_KINDS:
        s = f"a {ent['kind']} with {a.get('hair_color')} hair wearing a {a.get('clothing_color')} {a.get('clothing')}"
    else:
        s = f"a {a.get('color', '')} {ent['kind']}".replace("  ", " ")
        if a.get("state") == "open":
            s = f"an open {a.get('color', '')} {ent['kind']}"
    if ent["held_by"]:
        holder = snap["entities"][ent["held_by"]]
        s += f" held by the {holder['kind']}"
    return s


def setting(env):
    s = f"in a {env['location']}" if env["location"] not in ("street",) else "on a city street"
    if env["weather"] in ("rain", "snow") and not env["indoors"]:
        s += f", {'rainy' if env['weather'] == 'rain' else 'snowy'} weather"
    if env["time_of_day"] != "day":
        s += f", at {env['time_of_day']}"
    return s


orch, base = [], []
for sc, snap in zip(plan["scenes"], snaps):
    ents = snap["entities"]
    chars = [look(e, snap) for e in ents.values() if e["type"] == "character"]
    objs = [look(e, snap) for e in ents.values() if e["type"] == "object"]
    orch.append({"prompt": fit([STYLE, setting(snap["env"]), sc["text"].rstrip(".")] + chars + objs),
                 "negative": fit([BASE_NEGATIVE, "different clothes", "different hair color", "extra people"]),
                 "location": snap["env"]["location"]})
    base.append({"prompt": fit([STYLE, sc["text"].rstrip(".")]), "negative": BASE_NEGATIVE})

display(pd.DataFrame([{"#": k + 1, "baseline prompt": b["prompt"], "orchestrator prompt": o["prompt"]}
                      for k, (b, o) in enumerate(zip(base, orch))]))
with open(f"{OUT}/prompts.json", "w") as fh:
    json.dump({"baseline": base, "orchestrated": orch}, fh, indent=2)

## 4. Stage 4b: generate both videos with AnimateDiff
Same model, same settings, same number of scenes. The only difference is the conditioning:

| | prompt | seed | IP-Adapter (visual memory) |
|---|---|---|---|
| Baseline | scene sentence | new per scene | off |
| Orchestrator | state-graph prompt | fixed | last frame in this place / scene-1 anchor |

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, DDIMScheduler, MotionAdapter

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
adapter = MotionAdapter.from_pretrained(MOTION_ADAPTER, torch_dtype=dtype)
pipe = AnimateDiffPipeline.from_pretrained(BASE_MODEL, motion_adapter=adapter, torch_dtype=dtype)
pipe.scheduler = DDIMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler", clip_sample=False,
                                               timestep_spacing="linspace", beta_schedule="linear", steps_offset=1)
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
pipe.vae.enable_slicing()  # decode frames one at a time to save GPU memory
pipe.to(device)
pipe.set_progress_bar_config(disable=True)
print("loaded on", device)

In [ ]:
import time

BLANK = PILImage.new("RGB", (224, 224), (127, 127, 127))


def generate(prompt, negative, seed, ref, ip_scale):
    pipe.set_ip_adapter_scale(ip_scale)
    out = pipe(prompt=prompt, negative_prompt=negative, num_frames=FRAMES, guidance_scale=GUIDANCE,
               num_inference_steps=STEPS, height=SIZE, width=SIZE, ip_adapter_image=ref,
               generator=torch.Generator("cpu").manual_seed(seed))
    # diffusers returns RGB PIL frames; the CV code in this repo works in BGR
    return [cv2.cvtColor(np.array(f), cv2.COLOR_RGB2BGR) for f in out.frames[0]]


def to_pil(bgr):
    return PILImage.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))


clips = {"baseline": [], "orchestrated": []}
place_memory, anchor = {}, None
t0 = time.time()
for k in range(len(plan["scenes"])):
    clips["baseline"].append(generate(base[k]["prompt"], base[k]["negative"], SEED + 7919 * (k + 1), BLANK, 0.0))

    loc = orch[k]["location"]
    if anchor is None:
        ref, scale = BLANK, 0.0                       # scene 1 sets the reference look
    elif loc in place_memory:
        ref, scale = place_memory[loc], IP_SCALE      # back in a known place: continue from what it looked like
    else:
        ref, scale = anchor, IP_SCALE * 0.6           # new place: keep the characters, let the text set the room
    frames = generate(orch[k]["prompt"], orch[k]["negative"], SEED, ref, scale)
    clips["orchestrated"].append(frames)
    if anchor is None:
        anchor = to_pil(frames[len(frames) // 2])
    place_memory[loc] = to_pil(frames[-1])
    print(f"scene {k + 1}/{len(plan['scenes'])} done ({time.time() - t0:.0f} s)")

### Generated videos

In [ ]:
FPS = 8


def write_clips(arm):
    os.makedirs(f"{OUT}/{arm}", exist_ok=True)
    paths = []
    for k, frames in enumerate(clips[arm]):
        vw = VideoWriter(f"{OUT}/{arm}/scene{k + 1}", FPS, (SIZE, SIZE))
        for f in frames:
            vw.write(f)
        vw.close()
        paths.append(vw.path)
    return paths


def caption(img, text, color):
    img = img.copy()
    cv2.rectangle(img, (0, 0), (img.shape[1], 26), (25, 25, 25), -1)
    cv2.rectangle(img, (0, 24), (img.shape[1], 28), color, -1)
    cv2.putText(img, text, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (240, 240, 240), 1, cv2.LINE_AA)
    return img


paths = {arm: write_clips(arm) for arm in clips}
vw = VideoWriter(f"{OUT}/side_by_side", FPS, (2 * SIZE + 6, SIZE))
for k in range(len(plan["scenes"])):
    for fb, fo in zip(clips["baseline"][k], clips["orchestrated"][k]):
        vw.write(np.hstack([caption(fb, f"Baseline - scene {k + 1}", (52, 104, 235)),
                            np.full((SIZE, 6, 3), 20, np.uint8),
                            caption(fo, f"Orchestrator - scene {k + 1}", (214, 120, 42))]))
vw.close()
data = base64.b64encode(open(vw.path, "rb").read()).decode()
mime = "video/mp4" if vw.path.endswith(".mp4") else "video/webm"
display(HTML(f'<video controls loop autoplay muted width="100%"><source src="data:{mime};base64,{data}" '
             f'type="{mime}"></video><p>Left: baseline. Right: orchestrator. (If the player is empty, see the '
             f'storyboard below.)</p>'))

In [ ]:
def storyboard(rows, titles, fname, boxes=None):
    n = len(plan["scenes"])
    fig, axes = plt.subplots(len(rows), n, figsize=(2.6 * n, 2.9 * len(rows)), squeeze=False)
    for r, (arm, title) in enumerate(zip(rows, titles)):
        for k in range(n):
            ax = axes[r][k]
            f = clips[arm][k][len(clips[arm][k]) // 2]
            ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
            if boxes:
                for name, (x0, y0, x1, y1), ok in boxes[arm][k]:
                    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, lw=1.6,
                                               color="#3fbf5f" if ok else "#e04040"))
                    ax.text(x0 + 2, y0 + 12, name, fontsize=6.5, color="white",
                            bbox=dict(fc="#222", ec="none", pad=1))
            ax.set_xticks([]); ax.set_yticks([])
            if r == 0:
                ax.set_title(f"Scene {k + 1}", fontsize=10)
            if k == 0:
                ax.set_ylabel(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(f"{OUT}/{fname}", dpi=90)
    plt.close(fig)
    display(Image(f"{OUT}/{fname}"))  # the repo modules switch matplotlib to Agg, so show the saved file


storyboard(["baseline", "orchestrated"], ["Baseline", "Orchestrator"], "storyboard.png")
for k, sc in enumerate(plan["scenes"]):
    print(f"Scene {k + 1}: {sc['text']}")

## 5. Evaluation: baseline vs. orchestrator
Real generated video has no ground-truth masks, so pretrained vision models do the measuring:

* **OWLv2** (open-vocabulary detector) finds every character/object the state graph expects in each scene.
* **DINOv2** embeddings of each character's crop are compared with its first appearance (identity drift).
* **CLIP** checks which place each scene looks like, and how well the frame matches the scene text.
* **Farneback optical flow** (HW6) gives the warping error inside clips and across same-place cuts.

The six axes match the web app: character / object / spatial / temporal consistency, prompt adherence,
narrative flow.

In [ ]:
del pipe, adapter
if device == "cuda":
    torch.cuda.empty_cache()

from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor, pipeline as hf_pipeline

detector = hf_pipeline("zero-shot-object-detection", model="google/owlv2-base-patch16-ensemble",
                       device=0 if device == "cuda" else -1)
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dino = AutoModel.from_pretrained("facebook/dinov2-small").to(device).eval()
clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
DET_THRESHOLD = 0.15


@torch.no_grad()
def dino_embed(bgr):
    x = dino_proc(images=to_pil(bgr), return_tensors="pt").to(device)
    v = dino(**x).pooler_output[0]
    return (v / v.norm()).cpu().numpy()


@torch.no_grad()
def clip_scores(bgr, texts):
    x = clip_proc(text=texts, images=to_pil(bgr), return_tensors="pt", padding=True, truncation=True).to(device)
    out = clip(**x)
    img = out.image_embeds / out.image_embeds.norm(dim=-1, keepdim=True)
    txt = out.text_embeds / out.text_embeds.norm(dim=-1, keepdim=True)
    return (img @ txt.T)[0].cpu().numpy()


def detect(bgr, eids):
    # one query per entity kind; keep the best box per entity
    kinds = {eid: plan_ents[eid]["kind"] for eid in eids}
    labels = sorted(set(f"a {k}" for k in kinds.values()))
    found = {}
    if labels:
        for d in detector(to_pil(bgr), candidate_labels=labels, threshold=DET_THRESHOLD):
            for eid, kind in kinds.items():
                if d["label"] == f"a {kind}" and d["score"] > found.get(eid, (0,))[0]:
                    b = d["box"]
                    found[eid] = (d["score"], (b["xmin"], b["ymin"], b["xmax"], b["ymax"]))
    return found


plan_ents = {e["id"]: e for e in plan["entities"]}
LOCS = list(planner.KNOWN_LOCATIONS)
LOC_TEXT = [{"street": "a photo of a city street", "room": "a photo of a living room"}.get(l, f"a photo of a {l}")
            for l in LOCS]

In [ ]:
def evaluate(arm):
    frames = clips[arm]
    n = len(frames)
    mids = [f[len(f) // 2] for f in frames]
    det = [detect(mids[k], snaps[k]["visible"]) for k in range(n)]
    boxes = [[(plan_ents[e]["name"], det[k][e][1], True) for e in det[k]] for k in range(n)]

    # character identity vs. first appearance (re-anchored when the story changes the look)
    ident = []
    for eid, ent in plan_ents.items():
        if ent["type"] != "character":
            continue
        ref = None
        for k in range(n):
            if any(c["entity"] == eid for c in plan["scenes"][k]["attribute_changes"]):
                ref = None
            if eid not in det[k]:
                continue
            x0, y0, x1, y1 = [int(v) for v in det[k][eid][1]]
            crop = mids[k][max(0, y0):y1, max(0, x0):x1]
            if crop.size < 300:
                continue
            emb = dino_embed(crop)
            if ref is None:
                ref = emb
            else:
                ident.append({"character": ent["name"], "scene": k + 1, "dino_cos": float(ref @ emb)})
    character = float(np.mean([r["dino_cos"] for r in ident])) if ident else None

    # objects the state graph expects vs. objects the detector finds
    exp_obj = [(k, e) for k in range(n) for e in snaps[k]["visible"] if plan_ents[e]["type"] == "object"]
    obj = float(np.mean([e in det[k] for k, e in exp_obj])) if exp_obj else None
    vanished = [f"scene {k + 1}: {plan_ents[e]['name']}" for k, e in exp_obj
                if k > 0 and e not in det[k] and e in det[k - 1]]

    # same place across a cut: whole-frame DINO similarity + optical-flow warping error
    same = [k for k in range(1, n) if snaps[k]["env"]["location"] == snaps[k - 1]["env"]["location"]
            and snaps[k]["env"]["weather"] == snaps[k - 1]["env"]["weather"]
            and snaps[k]["env"]["time_of_day"] == snaps[k - 1]["env"]["time_of_day"]]
    spatial = float(np.mean([dino_embed(frames[k - 1][-1]) @ dino_embed(frames[k][0]) for k in same])) if same else None
    w_in = np.mean([metrics._warp_error(f[i], f[i + 1]) for f in frames for i in range(0, len(f) - 1, 2)])
    w_cut = np.mean([metrics._warp_error(frames[k - 1][-1], frames[k][0]) for k in same]) if same else None
    temporal = float(np.mean([max(0, 1 - w_in / 25)] + ([max(0, 1 - w_cut / 25)] if same else [])))

    # adherence: the scene's own entities (local) vs. the full expected state + place (narrative)
    p_ok = p_tot = n_ok = n_tot = 0
    clip_t, loc_seen = [], []
    for k, sc in enumerate(plan["scenes"]):
        loc_sim = clip_scores(mids[k], LOC_TEXT)
        seen = LOCS[int(np.argmax(loc_sim))]
        loc_seen.append(seen)
        clip_t.append(float(clip_scores(mids[k], [sc["text"]])[0]))
        p_checks = [e in det[k] for e in sc["present"] if e in snaps[k]["visible"]]
        p_ok += sum(p_checks); p_tot += len(p_checks)
        n_checks = [e in det[k] for e in snaps[k]["visible"]] + [seen == snaps[k]["env"]["location"]]
        n_ok += sum(n_checks); n_tot += len(n_checks)

    scores = {"character": character, "object": obj, "spatial": spatial, "temporal": temporal,
              "prompt": p_ok / p_tot if p_tot else None, "narrative": n_ok / n_tot if n_tot else None}
    valid = [v for v in scores.values() if v is not None]
    return {"scores": scores, "overall": float(np.mean(valid)), "identity": ident, "vanished": vanished,
            "clip_t": clip_t, "location_seen": loc_seen, "warp_in": float(w_in),
            "warp_cut": None if w_cut is None else float(w_cut), "boxes": boxes}


evals = {arm: evaluate(arm) for arm in clips}
pipeline._chart(evals, OUT, "scores.png")
display(Image(f"{OUT}/scores.png"))
display(pd.DataFrame({lbl: {("Baseline" if a == "baseline" else "Orchestrator"):
                             (None if evals[a]["scores"][key] is None else round(evals[a]["scores"][key], 3))
                             for a in evals} for key, lbl in metrics.AXES}).T)

### Evidence behind the scores

In [ ]:
storyboard(["baseline", "orchestrated"], ["Baseline", "Orchestrator"], "detections.png",
           boxes={a: evals[a]["boxes"] for a in evals})
rows = []
for k, snap in enumerate(snaps):
    for arm in ("baseline", "orchestrated"):
        found = {n for n, _, _ in evals[arm]["boxes"][k]}
        missing = [plan_ents[e]["name"] for e in snap["visible"] if plan_ents[e]["name"] not in found]
        rows.append({"scene": k + 1, "arm": arm, "expected place": snap["env"]["location"],
                     "CLIP sees": evals[arm]["location_seen"][k], "missing (expected by state graph)": ", ".join(missing),
                     "CLIP text-frame sim": round(evals[arm]["clip_t"][k], 3)})
display(pd.DataFrame(rows))
display(pd.DataFrame([dict(r, arm=a) for a in evals for r in evals[a]["identity"]]))
for a in evals:
    print(f"{a:13s} warping error in clips {evals[a]['warp_in']:.2f}, at same-place cuts {evals[a]['warp_cut']}; "
          f"objects vanishing: {evals[a]['vanished'] or 'none'}")

### Model-free measurements (same as the web app's *Evaluate real clips* page)

In [ ]:
free = {a: real_eval.evaluate_arm(clips[a], plan) for a in clips}
display(pd.DataFrame({lbl: {a: (None if free[a]["scores"][k] is None else round(free[a]["scores"][k], 3))
                            for a in free} for k, lbl in real_eval.REAL_AXES}).T)

### Video-LLM judge (Gemini)
Runs only if the `GEMINI_API_KEY` secret is set. It sees 3 frames per scene plus the story and the expected world
state, and scores each axis from 1 to 5.

In [ ]:
judged = judge.judge_runs(clips, plan, snaps)
if judged["error"] and not judged["arms"]:
    print(judged["error"])
else:
    display(pd.DataFrame({lbl: vals for lbl, vals in judge.as_rows(judged)}).T)
    for a, r in judged["arms"].items():
        print(f"\n{a}: {r['summary']}")
        for i in r["issues"]:
            print("  -", i)

## 6. Reference: the same story through the offline stand-in generator
This is what the web app runs without a GPU: the same planner, state graph and adapter, with a procedural
renderer as the generator.

In [ ]:
offline = pipeline.run(STORY, OUT, "offline", frames_per_scene=24)
display(Image(f"{OUT}/{offline['files']['storyboard']}", width=1000))
display(Image(f"{OUT}/{offline['files']['chart']}", width=700))

## 7. Summary

In [ ]:
labels = dict(metrics.AXES)
better = [labels[k] for k in labels if evals["orchestrated"]["scores"][k] is not None
          and evals["baseline"]["scores"][k] is not None
          and evals["orchestrated"]["scores"][k] > evals["baseline"]["scores"][k]]
print(f"Story: {len(plan['scenes'])} scenes, {len(plan_ents)} tracked entities, planner = {plan_info['planner']}")
print(f"Generator: AnimateDiff ({BASE_MODEL}), {FRAMES} frames x {len(plan['scenes'])} scenes per arm")
print(f"Overall score - baseline {evals['baseline']['overall']:.3f}, orchestrator {evals['orchestrated']['overall']:.3f} "
      f"(change {evals['orchestrated']['overall'] - evals['baseline']['overall']:+.3f})")
print(f"Orchestrator higher on {len(better)}/6 axes: {', '.join(better) or 'none'}")
for key, lbl in metrics.AXES:
    b, o = evals["baseline"]["scores"][key], evals["orchestrated"]["scores"][key]
    if b is not None and o is not None:
        print(f"  {lbl:22s} {b:.3f} -> {o:.3f}  ({o - b:+.3f})")
print(f"Objects vanishing between scenes - baseline {len(evals['baseline']['vanished'])}, "
      f"orchestrator {len(evals['orchestrated']['vanished'])}")
with open(f"{OUT}/results.json", "w") as fh:
    json.dump({"plan": plan, "prompts": {"baseline": base, "orchestrated": orch},
               "scores": {a: {k: evals[a][k] for k in ("scores", "overall", "identity", "vanished", "clip_t",
                                                        "location_seen", "warp_in", "warp_cut")} for a in evals},
               "model_free": free, "judge": judged}, fh, indent=2, default=str)

**Reading the result.** The two arms differ only in conditioning: same model, same steps, same resolution. If the
orchestrator scores higher on character/object/spatial consistency and narrative flow, the state graph plus the
adapter carried information across scenes that plain re-prompting loses. Prompt adherence (each scene vs. its
own sentence) shows whether that came at a cost to following the individual prompts.

**Limitations.** AnimateDiff makes 2-second, 512 px clips, and SD 1.5 doesn't follow long prompts precisely.
IP-Adapter carries global look rather than exact identity, and OWLv2/DINOv2/CLIP are proxies for human
judgement. A stronger generator (CogVideoX, Wan) with box-level layout control could plug into the same
state-graph conditioning without retraining.

In [ ]:
# download everything (clips per scene can be uploaded on the web app's "Evaluate real clips" page)
import shutil
with open(f"{OUT}/plan.json", "w") as fh:
    json.dump(plan, fh, indent=2)
archive = shutil.make_archive("/content/challenge1_results" if os.path.isdir("/content") else f"{OUT}_results",
                              "zip", OUT)
print("saved", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass